# 05 — Scenario: Supply Disruption

**The situation:** a supplier shipment into the Europe DC is delayed. The interesting
question isn't *"where is the shipment"* — it's *which stores, customers, and revenue
are actually at risk* over the delay window.

This notebook proves: supply-chain risk translated into business exposure, VIP
customer prioritization, and mitigation options.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Investigate — the delayed shipment

In [2]:
timeline = con.execute("""
    SELECT po.po_line_id, po.style_id, sty.style_name, po.order_date,
           po.original_expected_receipt_date, po.revised_expected_receipt_date, po.actual_receipt_date,
           e.event_seq, e.event_type, e.event_date
    FROM silver.fact_purchase_order_line po
    JOIN silver.dim_style sty ON sty.style_id = po.style_id
    JOIN silver.fact_shipment_event e ON e.po_line_id = po.po_line_id
    WHERE po.is_delayed ORDER BY po.po_line_id, e.event_seq
""").df()
print(f"{timeline['po_line_id'].nunique()} PO lines delayed, supplier -> DC-EUCEN")
timeline.head(12)

15 PO lines delayed, supplier -> DC-EUCEN


,po_line_id,style_id,style_name,order_date,original_expected_receipt_date,revised_expected_receipt_date,actual_receipt_date,event_seq,event_type,event_date
0,PO-000065-L1,STY-OUT-0001,Anorak 0001,2025-08-25,2025-11-17,2025-11-29,NaT,1,Booked,2025-08-25
1,PO-000065-L1,STY-OUT-0001,Anorak 0001,2025-08-25,2025-11-17,2025-11-29,NaT,2,Departed-Origin,2025-10-12
2,PO-000065-L1,STY-OUT-0001,Anorak 0001,2025-08-25,2025-11-17,2025-11-29,NaT,3,Customs-Hold,2025-11-29
3,PO-000266-L1,STY-OUT-0053,Parka 0053,2025-08-25,2025-11-17,2025-11-29,NaT,1,Booked,2025-08-25
4,PO-000266-L1,STY-OUT-0053,Parka 0053,2025-08-25,2025-11-17,2025-11-29,NaT,2,Departed-Origin,2025-10-12
5,PO-000266-L1,STY-OUT-0053,Parka 0053,2025-08-25,2025-11-17,2025-11-29,NaT,3,Customs-Hold,2025-11-29
6,PO-001059-L1,STY-OUT-0258,Trench Coat 0258,2025-08-25,2025-11-17,2025-11-29,NaT,1,Booked,2025-08-25
7,PO-001059-L1,STY-OUT-0258,Trench Coat 0258,2025-08-25,2025-11-17,2025-11-29,NaT,2,Departed-Origin,2025-10-12
8,PO-001059-L1,STY-OUT-0258,Trench Coat 0258,2025-08-25,2025-11-17,2025-11-29,NaT,3,Customs-Hold,2025-11-29
9,PO-001542-L1,STY-OUT-0381,Overcoat 0381,2025-08-25,2025-11-17,2025-11-29,NaT,1,Booked,2025-08-25


## Investigate — revenue at risk by region

In [3]:
risk = con.execute("""
    SELECT region_code, SUM(on_hand_units) on_hand, AVG(weeks_of_supply) avg_weeks_of_supply,
           SUM(trailing_avg_weekly_sales * 3 * current_retail_price) revenue_at_risk_3wk
    FROM gold.supply_risk_exposure GROUP BY 1 ORDER BY 4 DESC
""").df()
fig = px.bar(risk, x="region_code", y="revenue_at_risk_3wk", color_discrete_sequence=[STATUS["serious"]],
             labels={"region_code": "Region", "revenue_at_risk_3wk": "Revenue at risk, 3-week window (USD)"})
style_fig(fig, "Revenue at risk from the DC-EUCEN delay, by region")
print(f"Total revenue at risk: ${risk['revenue_at_risk_3wk'].sum():,.0f}")

Total revenue at risk: $3,673,745

## Investigate — VIP exposure

In [4]:
vip = con.execute("""
    SELECT c.loyalty_tier, COUNT(DISTINCT c.customer_id) n_customers
    FROM silver.dim_customer c
    WHERE c.home_region IN (SELECT DISTINCT region_code FROM gold.supply_risk_exposure)
      AND c.loyalty_tier IN ('Gold','Platinum','Private Client')
    GROUP BY 1 ORDER BY 2 DESC
""").df()
vip

,loyalty_tier,n_customers
0,Gold,65107
1,Platinum,35512
2,Private Client,15341


## Simulate — mitigation options

In [5]:
total_risk = risk["revenue_at_risk_3wk"].sum()
options = pd.DataFrame([
    {"Option": "Do nothing (accept the delay)", "Revenue protected": 0, "Cost": 0, "Notes": "Full exposure realized"},
    {"Option": "Expedite freight (air vs. ocean)", "Revenue protected": round(total_risk * 0.7),
     "Cost": round(total_risk * 0.07), "Notes": "Fastest, most expensive"},
    {"Option": "Transfer from other DCs", "Revenue protected": round(total_risk * 0.45),
     "Cost": round(total_risk * 0.03), "Notes": "Limited by donor-DC stock"},
    {"Option": "Controlled stockout, VIP-priority allocation", "Revenue protected": round(total_risk * 0.3),
     "Cost": round(total_risk * 0.01), "Notes": "Protects top customers only"},
])
options

,Option,Revenue protected,Cost,Notes
0,Do nothing (accept the delay),0,0,Full exposure realized
1,Expedite freight (air vs. ocean),2571621,257162,"Fastest, most expensive"
2,Transfer from other DCs,1653185,110212,Limited by donor-DC stock
3,"Controlled stockout, VIP-priority allocation",1102123,36737,Protects top customers only


## Recommend

Expedite freight for the highest-revenue-at-risk regions, and apply VIP-priority
allocation of whatever stock is on hand today for the remainder — protecting the
Private Client / Platinum customer base identified above while the expedited
shipment is in transit.